# Offline reconstruction of hourly throughput loss

Generated offline from archived experiment inputs and published throughput summaries.
**The hourly values are constructed by calibration, not recovered measurements.**
The figure follows the sans-serif, white-grid, blue-bar style of paper Fig. 11.

The original hourly throughput series is unavailable. This construction preserves
the two published distributions' means, quartiles, and whisker endpoints and assigns
them to hours using the rank of mean occupancy in the archived trace.
The resulting 24-hour aggregate is **0.70734%, rounded to 0.71%**, by calibration.
It does not independently validate the experiment or establish actual hourly losses.
See [README.md](README.md) for sources, hashes, assumptions, and dependencies.


In [ ]:
from pathlib import Path
import importlib.util
import json
import pandas as pd

# Works from this result folder, the repository root, or an enclosing workspace.
candidates = [Path.cwd(), Path.cwd() / "results/throughput-loss",
              Path.cwd() / "Energy-Saver-Tests/results/throughput-loss"]
result_dir = next((path for path in candidates if (path / "plot.py").is_file()
                   and (path / "input/published_throughput_summary.json").is_file()), None)
if result_dir is None:
    raise FileNotFoundError("Open this notebook from the result folder or repository root.")
result_dir = result_dir.resolve()
spec = importlib.util.spec_from_file_location("hourly_throughput_plot", result_dir / "plot.py")
plot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot)
report = plot.generate(result_dir / "out")
print(json.dumps({key: report[key] for key in (
    "generation_provenance", "computed_daily_reduction_percent",
    "displayed_daily_reduction_percent")}, indent=2))


In [ ]:
hourly = pd.read_csv(result_dir / "out/hourly-throughput-loss.csv")
hourly[["hour_start", "synthetic_baseline_gbps", "synthetic_solution_gbps",
        "synthetic_hourly_reduction_percent"]]


## Figure

The blue bars show one compatible calibrated allocation of hourly losses.
The dashed line is the baseline-volume-weighted 24-hour aggregate,
`100 * (sum(B_h) - sum(S_h)) / sum(B_h)`.
These are different from per-user losses and demand deficits.


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(result_dir / "out/hourly-throughput-loss.png"), width=900))
